In [5]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np

# ------------------ Load CSV ------------------
df = pd.read_csv("Results-Pearson-PredictedVSActual/Uniform_Pearson_Summary.csv")

# ------------------ Pivot for heatmap (robust to duplicates) ------------------
heatmap_data = pd.pivot_table(
    df,
    index='Matrix',
    columns='Method',
    values='Pearson_r',
    aggfunc='mean'
)

# ------------------ Flatten any MultiIndex on rows/cols ------------------
def flatten_index(idx):
    return idx.map(lambda t: "_".join(map(str, t))) if isinstance(idx, pd.MultiIndex) else idx

heatmap_data.index = flatten_index(heatmap_data.index)
heatmap_data.columns = flatten_index(heatmap_data.columns)

# ------------------ Desired Sampling order ------------------
sampling_order = ["2Median", "Sampling5", "Sampling10"]

# Build annotations from (now flat) index
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))
annotations['Sampling'] = idx_str.str.split('_').str[0]
annotations['MaxSig']   = idx_str.str.split('_').str[-1].astype(int)

# Make Sampling categorical with desired order
annotations['Sampling'] = pd.Categorical(annotations['Sampling'], categories=sampling_order, ordered=True)

# ------------------ Sort rows by Sampling then MaxSig ------------------
heatmap_data = (
    heatmap_data
    .assign(Sampling=annotations['Sampling'], MaxSig=annotations['MaxSig'])
    .sort_values(['Sampling', 'MaxSig'])
    .drop(columns=['Sampling', 'MaxSig'])
)

# Recompute annotations aligned to sorted index
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))
annotations['Sampling'] = pd.Categorical(idx_str.str.split('_').str[0], categories=sampling_order, ordered=True)
annotations['MaxSig']   = idx_str.str.split('_').str[-1].astype(int)

# ------------------ Reorder columns (BayesPrism, MuSiC first) ------------------
cols = ["BayesPrism", "MuSiC"] + [c for c in heatmap_data.columns if c not in ["BayesPrism", "MuSiC"]]
cols = [c for c in cols if c in heatmap_data.columns]  # guard if any missing
heatmap_data = heatmap_data[cols]

# ------------------ Palettes & LUTs ------------------
sampling_palette = sns.color_palette("Set2", n_colors=len(sampling_order))
sampling_lut = dict(zip(sampling_order, sampling_palette))

maxsig_keys = sorted(annotations['MaxSig'].unique())
maxsig_palette = sns.color_palette("Set3", n_colors=len(maxsig_keys))
maxsig_lut = dict(zip(maxsig_keys, maxsig_palette))

# ------------------ Row colors (NO .map; avoid MultiIndex isna path) ------------------
sampling_series = pd.Series(annotations['Sampling'].astype(str).values, index=annotations.index)
maxsig_series   = pd.Series(annotations['MaxSig'].astype(int).values,   index=annotations.index)

row_colors = pd.DataFrame({
    'Sampling': [sampling_lut[s] for s in sampling_series],
    'MaxSig':   [maxsig_lut[int(k)] for k in maxsig_series]
}, index=annotations.index)[['Sampling', 'MaxSig']]

# ------------------ Plot clustermap ------------------
g = sns.clustermap(
    heatmap_data,
    row_colors=row_colors,
    col_cluster=False,
    row_cluster=False,
    cmap='coolwarm',
    center=0,
    figsize=(12, 8),
    annot=True,
    cbar_pos=None,
    linewidths=0,
    linecolor='black'
)

# Hide y-axis labels and ticks
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_yticklabels([])

# ------------------ Pearson colorbar ------------------
cbar_ax = g.fig.add_axes([0.88, 0.2, 0.03, 0.6])
vmin = np.nanmin(heatmap_data.to_numpy())
vmax = np.nanmax(heatmap_data.to_numpy())
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = g.fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Pearson r')

# ------------------ Legends (in desired order) ------------------
sampling_patches = [Patch(facecolor=sampling_lut[s], label=s) for s in sampling_order]
maxsig_patches = [Patch(facecolor=maxsig_lut[k], label=str(k)) for k in maxsig_keys]

plt.subplots_adjust(top=0.95, bottom=0.05, left=0.2, right=0.85)

# Sampling legend first
legend_ax1 = g.fig.add_axes([0.02, 0.65, 0.20, 0.20])  # moved left
legend_ax1.axis('off')
legend_ax1.legend(handles=sampling_patches, title='Sampling', loc='center', fontsize=8, title_fontsize=9)

# Max Signatures legend below
legend_ax2 = g.fig.add_axes([0.02, 0.45, 0.20, 0.20])  # moved left
legend_ax2.axis('off')
legend_ax2.legend(handles=maxsig_patches, title='Max Signatures', loc='center', fontsize=8, title_fontsize=9)

# ------------------ Save ------------------
plt.savefig("Heatmap_Uniform_Pearson_NoGridlines.svg", bbox_inches="tight")
plt.close()


In [6]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
from matplotlib.colors import Normalize
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# ------------------ Load CSV ------------------
df = pd.read_csv("Results-Pearson-PredictedVSActual/Random_Pearson_Summary.csv")

# ------------------ Pivot for heatmap ------------------
heatmap_data = pd.pivot_table(
    df,
    index='Matrix',
    columns='Method',
    values='Pearson_r',
    aggfunc='mean'
)

# ------------------ Flatten any MultiIndex ------------------
def flatten_index(idx):
    return idx.map(lambda t: "_".join(map(str, t))) if isinstance(idx, pd.MultiIndex) else idx

heatmap_data.index = flatten_index(heatmap_data.index)
heatmap_data.columns = flatten_index(heatmap_data.columns)

# ------------------ Desired Sampling order ------------------
sampling_order = ["2Median", "Sampling5", "Sampling10"]

# Build annotations
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))
annotations['Sampling'] = idx_str.str.split('_').str[0]
annotations['MaxSig']   = idx_str.str.split('_').str[-1].astype(int)
annotations['Sampling'] = pd.Categorical(annotations['Sampling'], categories=sampling_order, ordered=True)

# ------------------ Sort rows ------------------
heatmap_data = (
    heatmap_data
    .assign(Sampling=annotations['Sampling'], MaxSig=annotations['MaxSig'])
    .sort_values(['Sampling', 'MaxSig'])
    .drop(columns=['Sampling', 'MaxSig'])
)

# Recompute annotations after sorting
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))
annotations['Sampling'] = pd.Categorical(idx_str.str.split('_').str[0], categories=sampling_order, ordered=True)
annotations['MaxSig']   = idx_str.str.split('_').str[-1].astype(int)

# ------------------ Reorder columns ------------------
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
method_order = [c for c in method_order if c in heatmap_data.columns]
heatmap_data = heatmap_data.reindex(columns=method_order)

# ------------------ Palettes & LUTs ------------------
sampling_palette = sns.color_palette("Set2", n_colors=len(sampling_order))
sampling_lut = dict(zip(sampling_order, sampling_palette))

maxsig_keys = sorted(annotations['MaxSig'].unique())
maxsig_palette = sns.color_palette("Set3", n_colors=len(maxsig_keys))
maxsig_lut = dict(zip(maxsig_keys, maxsig_palette))

# ------------------ Row colors ------------------
sampling_series = pd.Series(annotations['Sampling'].astype(str).values, index=annotations.index)
maxsig_series   = pd.Series(annotations['MaxSig'].astype(int).values,   index=annotations.index)

row_colors = pd.DataFrame({
    'Sampling': [sampling_lut[s] for s in sampling_series],
    'MaxSig':   [maxsig_lut[int(k)] for k in maxsig_series]
}, index=annotations.index)[['Sampling', 'MaxSig']]

# ------------------ Fixed color normalization ------------------
norm = Normalize(vmin=-0.6, vmax=0.6)

# ------------------ Plot clustermap ------------------
g = sns.clustermap(
    heatmap_data,
    row_colors=row_colors,
    col_cluster=False,
    row_cluster=False,
    cmap='coolwarm',
    norm=norm,
    figsize=(8, 5),
    annot=True, fmt=".2f",
    annot_kws={"size": 8},
    cbar_pos=None,
    linewidths=0,
    linecolor='white',
    dendrogram_ratio=(0.02, 0.02)
)

# === Axis styling ===
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_yticklabels([])
g.ax_heatmap.set_ylabel("Matrix", labelpad=30, fontsize=10)
g.ax_heatmap.yaxis.set_label_position("left")

g.ax_heatmap.set_xlabel("Deconvolution Tool", fontsize=10)
g.ax_heatmap.tick_params(axis='x', bottom=True, labelbottom=True, length=3, width=0.5, labelsize=8)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=30, ha="right")

# === Colorbar ===
cbar_ax = g.fig.add_axes([0.88, 0.23, 0.03, 0.5])
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=norm)
sm.set_array([])
cbar = g.fig.colorbar(sm, cax=cbar_ax)
cbar.ax.set_ylabel('Pearson r', fontsize=10, rotation=270, labelpad=15)   
cbar.ax.tick_params(labelsize=6, width=0.4, length=3)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === Legends ===
sampling_patches = [Patch(facecolor=sampling_lut[s], label=s) for s in sampling_order]
maxsig_patches = [Patch(facecolor=maxsig_lut[k], label=str(k)) for k in maxsig_keys]

legend_ax1 = g.fig.add_axes([0.02, 0.60, 0.2, 0.20])
legend_ax1.axis('off')
legend_ax1.legend(handles=sampling_patches, title='Sampling', loc='center', fontsize=8, title_fontsize=9)

legend_ax2 = g.fig.add_axes([0.02, 0.40, 0.2, 0.20])
legend_ax2.axis('off')
legend_ax2.legend(handles=maxsig_patches, title='Max Signatures', loc='center', fontsize=8, title_fontsize=9)

g.ax_row_colors.set_xticks([])
g.ax_row_colors.set_yticks([])
g.ax_row_colors.set_xticklabels([])
g.ax_row_colors.set_yticklabels([])

plt.subplots_adjust(top=0.95, bottom=0.05, left=0.25, right=0.85)
#plt.show()
g.fig.savefig("Heatmap_Random_Pearson_Final.svg", bbox_inches="tight")
plt.close(g.fig)

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
from matplotlib.colors import Normalize
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# ------------------ Load CSV ------------------
df = pd.read_csv("Results-Pearson-PredictedVSActual/Uniform_Pearson_Summary.csv")

# ------------------ Pivot for heatmap ------------------
heatmap_data = pd.pivot_table(
    df,
    index='Matrix',
    columns='Method',
    values='Pearson_r',
    aggfunc='mean'
)

# ------------------ Flatten any MultiIndex ------------------
def flatten_index(idx):
    return idx.map(lambda t: "_".join(map(str, t))) if isinstance(idx, pd.MultiIndex) else idx

heatmap_data.index = flatten_index(heatmap_data.index)
heatmap_data.columns = flatten_index(heatmap_data.columns)

# ------------------ Desired Sampling order ------------------
sampling_order = ["2Median", "Sampling5", "Sampling10"]

# Build annotations
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))
annotations['Sampling'] = idx_str.str.split('_').str[0]
annotations['MaxSig']   = idx_str.str.split('_').str[-1].astype(int)
annotations['Sampling'] = pd.Categorical(annotations['Sampling'], categories=sampling_order, ordered=True)

# ------------------ Sort rows ------------------
heatmap_data = (
    heatmap_data
    .assign(Sampling=annotations['Sampling'], MaxSig=annotations['MaxSig'])
    .sort_values(['Sampling', 'MaxSig'])
    .drop(columns=['Sampling', 'MaxSig'])
)

# Recompute annotations after sorting
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))
annotations['Sampling'] = pd.Categorical(idx_str.str.split('_').str[0], categories=sampling_order, ordered=True)
annotations['MaxSig']   = idx_str.str.split('_').str[-1].astype(int)

# ------------------ Reorder columns ------------------
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
heatmap_data = heatmap_data.reindex(columns=method_order)

# ------------------ Palettes & LUTs ------------------
sampling_palette = sns.color_palette("Set2", n_colors=len(sampling_order))
sampling_lut = dict(zip(sampling_order, sampling_palette))

maxsig_keys = sorted(annotations['MaxSig'].unique())
maxsig_palette = sns.color_palette("Set3", n_colors=len(maxsig_keys))
maxsig_lut = dict(zip(maxsig_keys, maxsig_palette))

# ------------------ Row colors ------------------
sampling_series = pd.Series(annotations['Sampling'].astype(str).values, index=annotations.index)
maxsig_series   = pd.Series(annotations['MaxSig'].astype(int).values,   index=annotations.index)

row_colors = pd.DataFrame({
    'Sampling': [sampling_lut[s] for s in sampling_series],
    'MaxSig':   [maxsig_lut[int(k)] for k in maxsig_series]
}, index=annotations.index)[['Sampling', 'MaxSig']]

# ------------------ Fixed color normalization ------------------
norm = Normalize(vmin=-0.6, vmax=0.6)

# ------------------ Plot clustermap ------------------
g = sns.clustermap(
    heatmap_data,
    row_colors=row_colors,
    col_cluster=False,
    row_cluster=False,
    cmap='coolwarm',
    norm=norm,
    figsize=(8, 5),
    annot=True, fmt=".2f",
    annot_kws={"size": 11},
    cbar_pos=None,
    linewidths=0,
    linecolor='white',
    dendrogram_ratio=(0.02, 0.02)
)

# === Axis styling ===
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_yticklabels([])
g.ax_heatmap.set_ylabel("Matrix", labelpad=30, fontsize=13)
g.ax_heatmap.yaxis.set_label_position("left")

g.ax_heatmap.set_xlabel("Deconvolution Tool", fontsize=13)
g.ax_heatmap.tick_params(axis='x', bottom=True, labelbottom=True, length=3, width=0.5, labelsize=8)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=30, ha="right")

# === Colorbar ===
cbar_ax = g.fig.add_axes([0.88, 0.23, 0.03, 0.5])
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=norm)
sm.set_array([])
cbar = g.fig.colorbar(sm, cax=cbar_ax)
cbar.ax.set_ylabel('Pearson r', fontsize=13, rotation=270, labelpad=20)   
cbar.ax.tick_params(labelsize=12, width=0.4, length=4)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === Legends ===
sampling_patches = [Patch(facecolor=sampling_lut[s], label=s) for s in sampling_order]
maxsig_patches = [Patch(facecolor=maxsig_lut[k], label=str(k)) for k in maxsig_keys]

legend_ax1 = g.fig.add_axes([0.02, 0.65, 0.20, 0.20])  # moved left
legend_ax1.axis('off')
legend_ax1.legend(handles=sampling_patches, title='Sampling', loc='center', fontsize=11, title_fontsize=12)

legend_ax2 = g.fig.add_axes([0.02, 0.45, 0.20, 0.20])  # moved left
legend_ax2.axis('off')
legend_ax2.legend(handles=maxsig_patches, title='Max Signatures', loc='center', fontsize=11, title_fontsize=12)

g.ax_row_colors.set_xticks([])
g.ax_row_colors.set_yticks([])
g.ax_row_colors.set_xticklabels([])
g.ax_row_colors.set_yticklabels([])

plt.subplots_adjust(top=0.95, bottom=0.05, left=0.25, right=0.85)
g.fig.savefig("Heatmap_Uniform_Pearson_Final.svg")
plt.close(g.fig)